In [5]:
import os, yaml, sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import h5py, cv2, joblib
from torchvision.models.feature_extraction import create_feature_extractor
from IPython.display import clear_output
ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])
from useful_stuff.general_utils.utils import TimeSeries, print_wise
from useful_stuff.image_processing.utils import load_torchvision_model, load_timm_model, get_relevant_output_layers
from project_specific_utils.dataloader import load_eyetracking_data
from image_processing.utils import read_video
from image_processing.gaze_dep_models import sequential_gaze_dep_mod, save_ipca_patch, save_ANN_features, ANN_extraction_projection_1917_wrapper

In [2]:
from dataclasses import dataclass, field

@dataclass
class Cfg:
    sub_num = 3
    fs = 24
    model_name = "alexnet";
    layer_name = "features.0";
    pkg = "torchvision"
    sq_side = 384
cfg = Cfg()
layers = get_relevant_output_layers(cfg.model_name)
print(layers)

['features.0', 'features.4', 'features.7', 'features.9', 'features.11', 'classifier.2', 'classifier.5']


# TODO run function to see if it's working (listen to audio)

In [ ]:
from project_specific_utils.utils import run2part
from image_processing.gaze_dep_models import pad_frame, extract_square_patch, extract_features_1917_movie
from useful_stuff.image_processing.utils import get_video_dimensions, preprocess_batch
import torch

"""
gaze_dep_ANN_extraction
Extract gaze-dependent ANN features from a movie by sampling square patches
centered at gaze positions and projecting them onto PCA components.

INPUT:
    - paths: dict[str, str] -> Dictionary containing data and output paths.
    - rank: int -> Process rank (for logging).
    - sub_num: int -> Subject identifier.
    - sq_side: int -> Side length of the square patch (in pixels).
    - model: torch.nn.Module -> Pretrained model used for feature extraction.
    - model_name: str -> Name of the model (used for saving).
    - layer_name: str -> Model layer from which features are extracted.
    - n_components: int -> Number of PCA components.
    - pooling: str -> Pooling strategy applied to features.
    - PCs: np.ndarray -> PCA projection matrix (features projected via @ PCs).
    - input_size: int -> Input size expected by the model.
    - run: int -> Run index (1–6).
    - eye_fs: int -> Eyetracking sampling frequency.
    - device: str -> Device for computation ("cpu", "cuda", "mps").
    - screen_res: tuple[int, int] (default=(1080, 1920)) -> Screen resolution (H, W).
    - secs_to_skip: int (default=5) -> Seconds skipped at the beginning of the movie.

OUTPUT:
    - None -> Saves extracted feature matrix to disk as an HDF5 file:
        dataset name: "vecrep", shape (n_components, n_frames).

NOTES:
    - Each frame is padded to screen resolution and sampled at gaze coordinates.
    - Features are extracted per frame and projected onto PCA space.
    - If output already exists, computation is skipped.
"""
def gaze_dep_ANN_extraction(paths: dict[str: str], rank: int, sub_num: int, sq_side: int, model, model_name: str,layer_name, n_components, pooling, PCs, input_size,  run: int, eye_fs, device, screen_res=(1080, 1920), secs_to_skip=5,): 
    movie_part = run2part(run)
    movie_fn = f"{paths['data_path']}/stimuli/Project1917_movie_part{movie_part}_24Hz.mp4"
    cap = cv2.VideoCapture(movie_fn)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.set(cv2.CAP_PROP_POS_FRAMES, round(5*fps))
    h, w, frames_n = get_video_dimensions(cap)
    save_name = save_ANN_features(paths, f"{model_name}_{layer_name}", round(fps), sub_num, run, n_components, sq_side, pooling,)
    if os.path.exists(save_name):
        print_wise(f"model already exists at {save_name}", rank=rank)
        return None
    # end if os.path.exists(save_name):
    feature_extractor = create_feature_extractor(
        model, return_nodes=[layer_name]
    ).to(device)
    xy_gaze, _ = load_eyetracking_data(paths, sub_num, run, eye_fs, xy=True)
    xy_gaze.resample(fps)
    frames_n -= round(secs_to_skip*fps) +2 # to be on the safe side, because when downsampled, the number of gaze-datapoints exceeds the number of frames
    if frames_n > len(xy_gaze):
        raise IndexError(f"The number of frames ({frames_n}) is larger than the number of gaze datapoints ({len(xy_gaze)}) in sub {sub_num} run {run}")
    # end if frames_n > len(xy_gaze):

    offset_dims = ((screen_res[0] -h)//2 , ( screen_res[1] - w)//2)
    canvas = None
    features = []
    for frame_idx in range(frames_n):
        xy = xy_gaze[frame_idx]
        ret, frame = cap.read()
        if not ret:
            raise RuntimeError(f"Failed to read frame {frame_idx} from {movie_fn}")
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        canvas = pad_frame(frame, (h,w), offset_dims,)
        frame_patch = extract_square_patch(canvas, round(xy[0]), round(xy[1]), sq_side)
        frame_patch = torch.from_numpy(frame_patch)
        curr_features = extract_features_1917_movie(frame_patch[None, :, :, :], feature_extractor, layer_name, input_size, pooling=pooling, device=device)
        curr_features = np.squeeze(curr_features @ PCs) # TODO check if features are a column vector
        features.append(curr_features)
        if frame_idx%1000 == 0:
            print_wise(f"processed frame {frame_idx} of {frames_n} in run {run} of {layer_name}")
        # canvas = sequential_gaze_dep_loop(cap, xy_gaze, frame_idx, sq_side, (h,w), offset_dims, canvas, features, func, rank, sub_num, run, *args, **kwargs)
    # end for frame_idx in range(frames_n):
    
    features = np.stack(features, axis=1)
    with h5py.File(save_name, "w") as f:
        f.create_dataset("vecrep", data=features)
    # end with h5py.File(save_name, "w") as f:
    print_wise(f"model {model_name} saved at {save_name}", rank=rank)
# EOF



"""
ANN_extraction_projection_1917_wrapper
Wrapper function to run gaze-dependent ANN feature extraction across all runs
using precomputed PCA components.

INPUT:
    - paths: dict[str, str] -> Dictionary containing data and output paths.
    - rank: int -> Process rank (for logging).
    - sub_num: int -> Subject identifier.
    - model: torch.nn.Module -> Pretrained model used for feature extraction.
    - sq_side: int -> Side length of the square patch (in pixels).
    - input_size: int -> Input size expected by the model.
    - model_name: str -> Name of the model.
    - layer_name: str -> Model layer from which features are extracted.
    - n_components: int -> Number of PCA components.
    - pooling: str -> Pooling strategy applied to features.
    - eye_fs: int -> Eyetracking sampling frequency.
    - device: str -> Device for computation ("cpu", "cuda", "mps").
    - screen_res: tuple[int, int] (default=(1080, 1920)) -> Screen resolution (H, W).
    - secs_to_skip: int (default=5) -> Seconds skipped at the beginning of each run.

OUTPUT:
    - None -> Calls `gaze_dep_ANN_extraction` for each run (1–6) and saves results to disk.

NOTES:
    - PCA components are loaded once and reused across runs.
    - Projection matrix is transposed before use.
"""
def ANN_extraction_projection_1917_wrapper(paths: dict[str: str], rank: int, sub_num: int, model, sq_side: int, input_size, model_name: str, layer_name, n_components, pooling, eye_fs, device, screen_res=(1080, 1920), secs_to_skip=5,): 
    ipca_path = save_ipca_patch(paths, model_name, layer_name, n_components, sq_side, pooling,) # model_name, layer_name, n_components, sq_size, pooling
    ipca_obj = joblib.load(ipca_path)
    PCs = ipca_obj.components_.T
    print_wise(f"Start running {model_name} for sub {sub_num}", rank=rank)
    for irun in range(1, 7):
        gaze_dep_ANN_extraction(paths, rank, sub_num, sq_side, model, model_name, layer_name, n_components, pooling, PCs, input_size, irun, eye_fs, device, screen_res=(1080, 1920), secs_to_skip=5, )
    # end for irun in range(1, 7):
# EOF



In [3]:
if cfg.pkg =="torchvision":
    model = load_torchvision_model(cfg.model_name, device='mps')
elif cfg.pkg == "timm":
    model = load_timm_model(cfg.model_name, device='mps')

In [ ]:
rank = 0; sub_num = 3;  n_components = 1000; pooling = "all"; input_size = 224; run = 1; eye_fs =50; device = "mps"

# gaze_dep_ANN_extraction(paths, rank, sub_num, cfg.sq_side, model, cfg.model_name, cfg.layer_name, n_components, pooling, PCs, input_size,  run, eye_fs, device, screen_res=(1080, 1920), secs_to_skip=5,)
for l in layers:
    ANN_extraction_projection_1917_wrapper(paths, rank, sub_num, model, cfg.sq_side, input_size, cfg.model_name, l, n_components, pooling, eye_fs, device,)

17:05:17 - rank 0 Start running alexnet for sub 3
17:05:17 - rank 0 model already exists at /Users/tizianocausin/1917_local/models/sub003_run01_alexnet_features.0_1000components_allpooling_gazedep_384x384_24Hz.h5
17:05:17 - rank 0 model already exists at /Users/tizianocausin/1917_local/models/sub003_run02_alexnet_features.0_1000components_allpooling_gazedep_384x384_24Hz.h5
17:05:17 - rank 0 model already exists at /Users/tizianocausin/1917_local/models/sub003_run03_alexnet_features.0_1000components_allpooling_gazedep_384x384_24Hz.h5
17:05:17 - rank 0 model already exists at /Users/tizianocausin/1917_local/models/sub003_run04_alexnet_features.0_1000components_allpooling_gazedep_384x384_24Hz.h5
17:05:17 - rank 0 model already exists at /Users/tizianocausin/1917_local/models/sub003_run05_alexnet_features.0_1000components_allpooling_gazedep_384x384_24Hz.h5
17:05:17 - rank 0 model already exists at /Users/tizianocausin/1917_local/models/sub003_run06_alexnet_features.0_1000components_allpool